# BBL opposition scouting pipeline

**The question:** before a T20 match, which bowlers should we use against their best
batters?

Cricsheet ball-by-ball → PostgreSQL star schema → 4 views → Power BI (3 pages).
**662 matches · 153,250 deliveries · 2011/12–2025/26**

Run top to bottom. Every stage validates itself and halts on failure. **~2 minutes**, plus
one R command in §6.

| | Section | Builds | What it holds |
|---|---|---|---|
| 1 | Load raw deliveries | `bronze_deliveries` | every ball, untouched |
| 2 | Standardise venues | `dim_venue` | 32 names → 21 real grounds |
| 3 | Match table | `dim_match` | one row per game |
| 4 | Player identity | `dim_player` | who each player actually is |
| 5 | Fact table | `fact_delivery` | one row per ball, fully linked |
| 6 | Bowling styles | `bowler_style` | each bowler's type |
| 7 | Serving views | 4 views | what the dashboard asks |

**Every decision is justified in `METHODOLOGY.md`.** Section numbers match; comments marked
`§n` point there.

**Inputs:** `data/bbl_csv/` (Cricsheet), `people.csv` (Cricsheet register).
`bowler_meta_all.csv` is generated in §6.

## 0 · Setup

The connection string reads from an environment variable so the notebook runs on any
machine without editing code. Set `BBL_DB_URL` first, or the local default applies.

The `check()` helper prints PASS or FAIL and **halts the notebook on failure** — bad data
cannot flow downstream unnoticed.

In [36]:
import os, glob, csv
import pandas as pd
from sqlalchemy import create_engine, text

DB_URL = os.getenv("BBL_DB_URL", "postgresql+psycopg2://harishthota@localhost:5432/bbl")
engine = create_engine(DB_URL)

def run_sql(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

def query(sql):
    return pd.read_sql(sql, engine)

def scalar(sql):
    return query(sql).iloc[0, 0]

def check(label, actual, expected=None, lo=None, hi=None):
    """Verify a value. Prints PASS/FAIL and raises on failure."""
    if expected is not None:
        ok, detail = actual == expected, f"{actual:,} (expected {expected:,})"
    else:
        ok, detail = lo <= actual <= hi, f"{actual:,} (expected {lo:,}-{hi:,})"
    print(f"{'PASS' if ok else 'FAIL'}  {label}: {detail}")
    assert ok, f"{label} failed: {detail}"

print("connected")

connected


Three thresholds decide what the analysis includes. Each was **measured against the data,
not assumed** — the trade-offs are in `METHODOLOGY.md` §7a.

In [37]:
DATA_DIR = "data/bbl_csv"

MATCHUP_MIN_BALLS  = 30    # head-to-head balls per batter-bowler pairing   §7a
BASELINE_MIN_BALLS = 500   # career balls before a batter's baseline holds  §7a
TYPE_MIN_BALLS     = 60    # balls against a bowling type                   §7c

print(f"matchup >= {MATCHUP_MIN_BALLS} | baseline >= {BASELINE_MIN_BALLS} | "
      f"type >= {TYPE_MIN_BALLS}")

matchup >= 30 | baseline >= 500 | type >= 60


## 1 · Load raw deliveries

**Build the file list first, carefully.** Cricsheet ships `all_matches.csv` next to the
662 individual match files, and it contains every ball already in them. Read the folder
naively and the whole dataset loads twice — silently, with no error.

The filter keeps only numerically-named match files. *(§1 — this bug survived two days of
validation before a cricket check caught it.)*

In [38]:
files = [f for f in glob.glob(f"{DATA_DIR}/*.csv")
         if not f.endswith("_info.csv")
         and os.path.basename(f).replace(".csv", "").isdigit()]

# report what was left out, so nothing is silently included or dropped
excluded = [os.path.basename(f) for f in glob.glob(f"{DATA_DIR}/*.csv")
            if f not in files and not f.endswith("_info.csv")]
print("excluded:", excluded or "none")

check("match files", len(files), expected=662)

excluded: ['all_matches.csv']
PASS  match files: 662 (expected 662)


Read every file into one table. Bronze stays raw — all cleaning happens later.

In [39]:
raw = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
check("rows in memory", len(raw), expected=153250)

raw.to_sql("bronze_deliveries", engine, if_exists="replace", index=False)
print("written to bronze_deliveries")

PASS  rows in memory: 153,250 (expected 153,250)
written to bronze_deliveries


**Check the database, not the dataframe.** During development these disagreed — Python
held 153,250 rows while the table still had 306,500, because the write had silently
failed. Verifying what you sent proves nothing about where you sent it. *(§1)*

In [40]:
b = query("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT match_id) AS matches,
       COUNT(DISTINCT (match_id, innings, ball, striker, bowler,
                       runs_off_bat, extras, wides, noballs)) AS distinct_rows,
       ROUND(COUNT(*)::numeric / COUNT(DISTINCT match_id), 0) AS per_match
FROM bronze_deliveries
""").iloc[0]

check("rows in database", int(b["rows"]),    expected=153250)
check("matches",          int(b["matches"]), expected=662)

# A T20 is 2 x 120 balls plus extras. 460 would mean the data doubled.
check("balls per match",  float(b["per_match"]), lo=200, hi=260)

# Tolerance is 2, not 0: a wide and its replacement share a ball number.  §1
check("exact duplicates", int(b["rows"]) - int(b["distinct_rows"]), lo=0, hi=2)

PASS  rows in database: 153,250 (expected 153,250)
PASS  matches: 662 (expected 662)
PASS  balls per match: 231.0 (expected 200-260)
PASS  exact duplicates: 1 (expected 0-2)


What a raw delivery row looks like — six consecutive balls from one over.

In [41]:
query("""SELECT match_id, innings, ball, batting_team, striker, bowler,
                runs_off_bat, extras, wicket_type
         FROM bronze_deliveries
         WHERE match_id = 524915 AND innings = 1
         ORDER BY ball LIMIT 6""")

,match_id,innings,ball,batting_team,striker,bowler,runs_off_bat,extras,wicket_type
0,524915,1,0.1,Brisbane Heat,BB McCullum,B Lee,0,0,None
1,524915,1,0.2,Brisbane Heat,BB McCullum,B Lee,0,0,None
2,524915,1,0.3,Brisbane Heat,BB McCullum,B Lee,4,0,None
3,524915,1,0.4,Brisbane Heat,BB McCullum,B Lee,0,0,None
4,524915,1,0.5,Brisbane Heat,BB McCullum,B Lee,0,0,None
5,524915,1,0.6,Brisbane Heat,BB McCullum,B Lee,0,0,None


## 2 · Standardise venues

**The problem:** 32 venue names in the data, but only **21 actual grounds**. Stadiums get
renamed when sponsors change, and the data records whatever the ground was called that
day. Left alone, a question like "how do teams score at the Gabba" splits three ways and
gives three incomplete answers.

First, create the table. It keeps the original name alongside a standardised one, so the
source is never destroyed.

In [42]:
run_sql("""
DROP TABLE IF EXISTS dim_venue CASCADE;
CREATE TABLE dim_venue (
    venue_id    SERIAL PRIMARY KEY,
    venue_name  TEXT UNIQUE NOT NULL,   -- exactly as the source wrote it
    venue_clean TEXT                    -- the standardised name we use
);
INSERT INTO dim_venue (venue_name)
SELECT DISTINCT venue FROM bronze_deliveries WHERE venue IS NOT NULL;

UPDATE dim_venue SET venue_clean = venue_name;   -- default: keep the original
""")

check("raw venue names", scalar("SELECT COUNT(*) FROM dim_venue"), expected=32)

PASS  raw venue names: 32 (expected 32)


**Two merges could not be found by matching text.** "Aurora Stadium" and "University of
Tasmania Stadium" share no words, but are the same ground — York Park in Launceston,
renamed when the sponsor changed in 2017. Same story for Simonds and GMHBA in Geelong.

Both verified against published venue histories rather than assumed. *(§2 cites the
sources.)*

In [43]:
run_sql("""
UPDATE dim_venue SET venue_clean='Aurora Stadium'
  WHERE venue_name IN ('Aurora Stadium, Launceston',
                       'University of Tasmania Stadium, Launceston');
UPDATE dim_venue SET venue_clean='Geelong Cricket Ground'
  WHERE venue_name IN ('GMHBA Stadium, South Geelong, Victoria',
                       'Simonds Stadium, South Geelong, Victoria');
""")
print("sponsor renames merged")

sponsor renames merged


The rest are the same name with a city appended, or an abbreviation.

In [44]:
run_sql("""
UPDATE dim_venue SET venue_clean='Bellerive Oval'
  WHERE venue_name='Bellerive Oval, Hobart';
UPDATE dim_venue SET venue_clean='Brisbane Cricket Ground'
  WHERE venue_name IN ('Brisbane Cricket Ground, Woolloongabba',
                       'Brisbane Cricket Ground, Woolloongabba, Brisbane');
UPDATE dim_venue SET venue_clean='Docklands Stadium'
  WHERE venue_name='Docklands Stadium, Melbourne';
UPDATE dim_venue SET venue_clean='International Sports Stadium'
  WHERE venue_name='International Sports Stadium, Coffs Harbour';
UPDATE dim_venue SET venue_clean='Manuka Oval'
  WHERE venue_name='Manuka Oval, Canberra';
UPDATE dim_venue SET venue_clean='WACA Ground'
  WHERE venue_name IN ('W.A.C.A. Ground',
                       'Western Australia Cricket Association Ground');
""")
print("location variants merged")

location variants merged


Validate, then list the grounds that had more than one spelling.

In [45]:
v = query("""SELECT COUNT(*) raw, COUNT(DISTINCT venue_clean) clean,
                    COUNT(*) FILTER (WHERE venue_clean IS NULL) blanks
             FROM dim_venue""").iloc[0]
check("raw names",    int(v["raw"]),    expected=32)
check("real grounds", int(v["clean"]),  expected=21)
check("blank names",  int(v["blanks"]), expected=0)

# every venue in the ball data needs a row here, or its matches vanish later
check("venues with no row",
      scalar("""SELECT COUNT(*) FROM (SELECT DISTINCT b.venue FROM bronze_deliveries b
                LEFT JOIN dim_venue v ON b.venue=v.venue_name
                WHERE v.venue_name IS NULL) x"""), expected=0)

query("""SELECT venue_clean, COUNT(*) AS spellings,
                STRING_AGG(venue_name, ' | ') AS variants
         FROM dim_venue GROUP BY venue_clean
         HAVING COUNT(*) > 1 ORDER BY spellings DESC""")

PASS  raw names: 32 (expected 32)
PASS  real grounds: 21 (expected 21)
PASS  blank names: 0 (expected 0)
PASS  venues with no row: 0 (expected 0)


,venue_clean,spellings,variants
0,Brisbane Cricket Ground,3,Brisbane Cricket Ground | Brisbane Cricket Gro...
1,Geelong Cricket Ground,3,"Geelong Cricket Ground | GMHBA Stadium, South ..."
2,Aurora Stadium,3,Aurora Stadium | University of Tasmania Stadiu...
3,Docklands Stadium,2,"Docklands Stadium | Docklands Stadium, Melbourne"
4,Manuka Oval,2,"Manuka Oval | Manuka Oval, Canberra"
5,WACA Ground,2,Western Australia Cricket Association Ground |...
6,International Sports Stadium,2,International Sports Stadium | International S...
7,Bellerive Oval,2,"Bellerive Oval | Bellerive Oval, Hobart"


## 3 · Match table

Collapses 153,250 ball records into **662 match records**. Date, teams and venue repeat on
every ball; here they are stored once.

**Note the column names.** An earlier version called these `team_home` and `team_away` —
an assumption that turned out wrong about half the time, because the coin toss decides who
bats first, not the fixture. They are named for what the data provably holds. *(§3)*

In [46]:
run_sql("""
DROP TABLE IF EXISTS dim_match CASCADE;
CREATE TABLE dim_match (
    match_id            BIGINT PRIMARY KEY,
    match_date          DATE,
    season              TEXT,
    team_batting_first  TEXT,        -- NOT home/away  §3
    team_bowling_first  TEXT,
    venue_id            INT REFERENCES dim_venue(venue_id)
);

-- teams swap between innings, so innings 1 identifies who batted first
INSERT INTO dim_match
SELECT DISTINCT b.match_id, b.start_date::date, b.season,
       b.batting_team, b.bowling_team, v.venue_id
FROM bronze_deliveries b
JOIN dim_venue v ON b.venue = v.venue_name
WHERE b.innings = 1;
""")
print("built")

built


Validate, then show what a match row contains.

In [47]:
m = query("""SELECT COUNT(*) rows, COUNT(DISTINCT match_id) ids,
                    COUNT(*) FILTER (WHERE match_date IS NULL OR venue_id IS NULL) nulls,
                    COUNT(*) FILTER (WHERE team_batting_first=team_bowling_first) self_match,
                    COUNT(DISTINCT team_batting_first) teams
             FROM dim_match""").iloc[0]

check("matches",              int(m["rows"]),       expected=662)
check("no duplicates",        int(m["ids"]),        expected=int(m["rows"]))
check("no missing values",    int(m["nulls"]),      expected=0)
check("no team plays itself", int(m["self_match"]), expected=0)
check("BBL teams",            int(m["teams"]),      expected=8)   # there are exactly 8

query("""SELECT m.match_id, m.match_date, m.season,
                m.team_batting_first, m.team_bowling_first, v.venue_clean
         FROM dim_match m JOIN dim_venue v ON m.venue_id = v.venue_id
         ORDER BY m.match_date LIMIT 5""")

PASS  matches: 662 (expected 662)
PASS  no duplicates: 662 (expected 662)
PASS  no missing values: 0 (expected 0)
PASS  no team plays itself: 0 (expected 0)
PASS  BBL teams: 8 (expected 8)


,match_id,match_date,season,team_batting_first,team_bowling_first,venue_clean
0,524915,2011-12-16,2011/12,Brisbane Heat,Sydney Sixers,Sydney Cricket Ground
1,524916,2011-12-17,2011/12,Melbourne Stars,Sydney Thunder,Melbourne Cricket Ground
2,524917,2011-12-18,2011/12,Adelaide Strikers,Melbourne Renegades,Adelaide Oval
3,524918,2011-12-18,2011/12,Hobart Hurricanes,Perth Scorchers,WACA Ground
4,524919,2011-12-20,2011/12,Melbourne Stars,Brisbane Heat,Brisbane Cricket Ground


## 4 · Player identity

**The problem:** the same player can be written different ways, and different players can
share a name. Get it wrong and you silently merge two careers or split one in half.

Cricsheet's match files carry a **registry** giving every player a permanent ID. Identity
comes from that ID, never from the name — two lookalike cases in this data resolved in
*opposite* directions, and name matching would have got both wrong. *(§4)*

First, extract the registry from all 662 info files.

In [48]:
info_files = glob.glob(f"{DATA_DIR}/*_info.csv")
check("info files", len(info_files), expected=662)

registry = {}
for path in info_files:
    with open(path) as f:
        for row in csv.reader(f):
            if len(row) >= 5 and row[1] == "registry" and row[2] == "people":
                registry.setdefault(row[3], set()).add(row[4])

reg_df = pd.DataFrame([{"registry_id": pid, "player_name": n}
                       for n, ids in registry.items() for pid in ids]).drop_duplicates()
reg_df.to_sql("registry_people", engine, if_exists="replace", index=False)

print(f"{reg_df.registry_id.nunique()} players, {reg_df.player_name.nunique()} names")

PASS  info files: 662 (expected 662)
639 players, 640 names


**The check that prevents silent duplication.** The fact table looks up each ball's batter
and bowler by name. If one name mapped to two IDs, that lookup would match twice and every
ball for that player would be duplicated, with no error to warn you.

An earlier version checked only the opposite direction — IDs with several names — which
misses this entirely. *(§4)*

In [49]:
dupes = query("""SELECT player_name, COUNT(DISTINCT registry_id) n
                 FROM registry_people GROUP BY player_name
                 HAVING COUNT(DISTINCT registry_id) > 1""")
check("names mapping to more than one ID", len(dupes), expected=0)
if len(dupes):
    print(dupes.to_string(index=False))

PASS  names mapping to more than one ID: 0 (expected 0)


Build the player table. Grouping by ID collapses a player written two ways into one row.

In [50]:
run_sql("""
DROP TABLE IF EXISTS dim_player CASCADE;
CREATE TABLE dim_player (
    registry_id TEXT PRIMARY KEY,
    player_name TEXT NOT NULL
);
INSERT INTO dim_player
SELECT registry_id, MAX(player_name) FROM registry_people GROUP BY registry_id;
""")

check("players", scalar("SELECT COUNT(*) FROM dim_player"),
      expected=int(scalar("SELECT COUNT(DISTINCT registry_id) FROM registry_people")))

# The two cases that prove ID beats name: the Greens stay apart, the Browns merged.  §4
query("""SELECT registry_id, player_name FROM dim_player
         WHERE player_name IN ('J Brown','Josh Brown','C Green','CJ Green')
         ORDER BY player_name""")

PASS  players: 639 (expected 639)


,registry_id,player_name
0,eaa76d3c,C Green
1,19b9f399,CJ Green
2,647e7d44,Josh Brown


Every player named in the ball data must resolve to an ID, or their deliveries get dropped.

In [51]:
for col in ["striker", "bowler", "non_striker"]:
    total   = scalar(f"SELECT COUNT(DISTINCT {col}) FROM bronze_deliveries")
    matched = scalar(f"""SELECT COUNT(DISTINCT b.{col}) FROM bronze_deliveries b
                         JOIN registry_people r ON b.{col}=r.player_name""")
    check(f"{col}s resolved to an ID", matched, expected=total)

PASS  strikers resolved to an ID: 503 (expected 503)
PASS  bowlers resolved to an ID: 372 (expected 372)
PASS  non_strikers resolved to an ID: 507 (expected 507)


## 5 · Fact table

The centre of the schema — **one row per delivery**, 153,250 rows. Every column is either
a **link** to a dimension or a **fact** about that ball. Nothing else belongs here.

Two derived columns worth knowing:

- **`bowler_wicket`** — true only when the bowler earned the wicket. Run-outs and
  retirements are excluded, because a run-out is a fielding event. Counting them would
  overstate every bowler's record. *(§5)*
- **`phase`** — Powerplay (overs 0–5), Middle (6–14), Death (15–19). Batters behave
  completely differently in each, so it is worked out once here.

In [52]:
run_sql("""
DROP TABLE IF EXISTS fact_delivery CASCADE;
CREATE TABLE fact_delivery (
    delivery_id   BIGSERIAL PRIMARY KEY,
    match_id      BIGINT REFERENCES dim_match(match_id),
    innings       INT,
    over_num      INT,
    phase         TEXT,        -- Powerplay | Middle | Death
    batter_id     TEXT REFERENCES dim_player(registry_id),
    bowler_id     TEXT REFERENCES dim_player(registry_id),
    runs_off_bat  INT,
    extras        INT,
    is_wicket     BOOLEAN,     -- any dismissal
    wicket_type   TEXT,
    bowler_wicket BOOLEAN      -- bowler-credited only  §5
);
""")
print("table created")

table created


Populate it, looking up each ball's batter and bowler IDs from their names.

In [53]:
run_sql("""
INSERT INTO fact_delivery
    (match_id, innings, over_num, phase, batter_id, bowler_id,
     runs_off_bat, extras, is_wicket, wicket_type, bowler_wicket)
SELECT b.match_id,
       b.innings,
       FLOOR(b.ball)::int,                      -- ball 5.3 sits in over 5
       CASE WHEN FLOOR(b.ball) < 6  THEN 'Powerplay'
            WHEN FLOOR(b.ball) < 15 THEN 'Middle'
            ELSE 'Death' END,
       rb.registry_id,                          -- batter ID, from his name
       rbo.registry_id,                         -- bowler ID
       b.runs_off_bat,
       COALESCE(b.extras, 0),                   -- blanks become 0
       (b.wicket_type IS NOT NULL),
       b.wicket_type,
       (b.wicket_type IS NOT NULL AND b.wicket_type NOT IN
        ('run out','retired hurt','retired out','obstructing the field'))
FROM bronze_deliveries b
JOIN registry_people rb  ON b.striker = rb.player_name
JOIN registry_people rbo ON b.bowler  = rbo.player_name;
""")
print(f"{scalar('SELECT COUNT(*) FROM fact_delivery'):,} deliveries loaded")

153,250 deliveries loaded


Six consecutive balls — one over — showing what a fact row holds. Note the IDs where the
names were, and the two derived columns on the right.

In [54]:
query("""SELECT delivery_id, match_id, innings, over_num, phase,
                batter_id, bowler_id, runs_off_bat, extras,
                wicket_type, bowler_wicket
         FROM fact_delivery
         WHERE match_id = 524915 AND innings = 1
         ORDER BY delivery_id LIMIT 6""")

,delivery_id,match_id,innings,over_num,phase,batter_id,bowler_id,runs_off_bat,extras,wicket_type,bowler_wicket
0,66099,524915,1,0,Powerplay,b8a55852,dd09ff8e,0,0,None,False
1,66100,524915,1,0,Powerplay,b8a55852,dd09ff8e,0,0,None,False
2,66101,524915,1,0,Powerplay,b8a55852,dd09ff8e,4,0,None,False
3,66102,524915,1,0,Powerplay,b8a55852,dd09ff8e,0,0,None,False
4,66103,524915,1,0,Powerplay,b8a55852,dd09ff8e,0,0,None,False
5,66104,524915,1,0,Powerplay,b8a55852,dd09ff8e,0,0,None,False


### 5a · Validation

Four families of check. **Completeness** and **referential integrity** compare stages to
each other. **Reality checks** compare the data to cricket itself — and that second kind is
essential, because the earlier doubling bug passed every relative check while both sides
were doubled.

In [55]:
f = query("""SELECT COUNT(*) rows, COUNT(DISTINCT delivery_id) ids,
                    COUNT(DISTINCT match_id) matches,
                    MIN(runs_off_bat) min_runs, MAX(runs_off_bat) max_runs,
                    ROUND(COUNT(*)::numeric/COUNT(DISTINCT match_id),0) per_match,
                    ROUND(SUM(runs_off_bat)::numeric/COUNT(DISTINCT match_id),0) runs_per_match
             FROM fact_delivery""").iloc[0]

print("completeness - nothing lost on the way in")
check("  rows match bronze", int(f["rows"]),
      expected=int(scalar("SELECT COUNT(*) FROM bronze_deliveries")))
check("  no duplicate rows", int(f["ids"]),     expected=int(f["rows"]))
check("  all matches kept",  int(f["matches"]), expected=662)

print("\nreferential integrity - every link points at something real")
for lbl, tbl, fk, pk in [("match", "dim_match", "match_id", "match_id"),
                         ("batter","dim_player","batter_id","registry_id"),
                         ("bowler","dim_player","bowler_id","registry_id")]:
    check(f"  orphan {lbl}s",
          scalar(f"""SELECT COUNT(*) FROM fact_delivery f
                     LEFT JOIN {tbl} d ON f.{fk}=d.{pk} WHERE d.{pk} IS NULL"""),
          expected=0)

print("\nvalidity - values in sensible ranges")
check("  min runs off one ball", int(f["min_runs"]), expected=0)
check("  max runs off one ball", int(f["max_runs"]), lo=6, hi=6)

print("\nreality - matches what cricket says should be true")
check("  balls per match", float(f["per_match"]),      lo=200, hi=260)
check("  runs per match",  float(f["runs_per_match"]), lo=240, hi=360)

completeness - nothing lost on the way in
PASS    rows match bronze: 153,250 (expected 153,250)
PASS    no duplicate rows: 153,250 (expected 153,250)
PASS    all matches kept: 662 (expected 662)

referential integrity - every link points at something real
PASS    orphan matchs: 0 (expected 0)
PASS    orphan batters: 0 (expected 0)
PASS    orphan bowlers: 0 (expected 0)

validity - values in sensible ranges
PASS    min runs off one ball: 0 (expected 0)
PASS    max runs off one ball: 6 (expected 6-6)

reality - matches what cricket says should be true
PASS    balls per match: 231.0 (expected 200-260)
PASS    runs per match: 289.0 (expected 240-360)


**Reconciliation.** The three phases must add up to the total. A shortfall would mean some
balls fell through the `CASE` and got no phase at all.

In [56]:
ph = query("""SELECT phase, COUNT(*) balls FROM fact_delivery
              GROUP BY phase ORDER BY balls DESC""")

check("phases sum to the total", int(ph.balls.sum()),
      expected=int(scalar("SELECT COUNT(*) FROM fact_delivery")))
print()
print(ph.to_string(index=False))

PASS  phases sum to the total: 153,250 (expected 153,250)

    phase  balls
   Middle  70903
Powerplay  48968
    Death  33379


### 5b · External check

Every check so far compares the data to itself, or to a general expectation. This one
compares a computed figure to a **published statistic** — Chris Lynn's BBL strike rate is
around 145.

**The ball count is checked too, and that is the point.** A doubled dataset would still
show a strike rate of 145, because runs and balls both double and the division cancels.
Only the ball count gives it away. *(§5b)*

In [57]:
l = query("""SELECT COUNT(*) balls, ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) sr
             FROM fact_delivery f JOIN dim_player p ON f.batter_id=p.registry_id
             WHERE p.player_name='CA Lynn'""").iloc[0]

check("CA Lynn strike rate vs published ~145", float(l["sr"]),   lo=138,  hi=152)
check("CA Lynn career balls, plausible",       int(l["balls"]),  lo=2000, hi=3600)

# one real match, both innings - should look like a T20
query("""SELECT innings, COUNT(*) balls, SUM(runs_off_bat) runs
         FROM fact_delivery WHERE match_id=524915 GROUP BY innings ORDER BY innings""")

PASS  CA Lynn strike rate vs published ~145: 145.4 (expected 138-152)
PASS  CA Lynn career balls, plausible: 2,812 (expected 2,000-3,600)


,innings,balls,runs
0,1,124,132
1,2,114,138


## 6 · Bowling styles

**Why this matters:** knowing a batter struggles against one particular bowler only helps
if you have that bowler. Knowing he struggles against **leg spin** works against any
opponent. It also turns 30-ball samples into 400-ball ones.

**The problem:** Cricsheet records what happened on each ball and carries no player
attributes at all. Styles come from ESPNCricinfo through the `cricketdata` R package —
scraping was tried and rejected, and the alternatives are compared in §6.

First, map our bowlers to their ESPNCricinfo IDs using Cricsheet's own register.

In [58]:
people = pd.read_csv("people.csv")

all_bowlers = query("""
    SELECT DISTINCT p.registry_id, p.player_name
    FROM fact_delivery f JOIN dim_player p ON f.bowler_id = p.registry_id
""")

# LEFT JOIN so anyone unmatched is visible rather than silently dropped
merged = all_bowlers.merge(people[['identifier','key_cricinfo']],
                           left_on='registry_id', right_on='identifier', how='left')
print("bowlers:", len(merged),
      "| matched to a Cricinfo ID:", merged['key_cricinfo'].notna().sum())

missing = merged[merged['key_cricinfo'].isna()]
if len(missing):
    print("\nno ID found for:\n", missing[['player_name']].to_string(index=False))

ids = merged['key_cricinfo'].dropna().astype(int).tolist()
with open("cricinfo_ids.txt", "w") as fh:
    fh.write("c(" + ", ".join(map(str, ids)) + ")")
print(f"\n{len(ids)} IDs written to cricinfo_ids.txt")

bowlers: 372 | matched to a Cricinfo ID: 372

372 IDs written to cricinfo_ids.txt


Write the R script that fetches the styles, so the step runs as a single command rather
than an interactive session.

**Then run this in your terminal:**

```
Rscript fetch_styles.R
```

First time only, install the package: `install.packages("devtools")` then
`devtools::install_github("robjhyndman/cricketdata")`.

In [59]:
with open("fetch_styles.R", "w") as fh:
    fh.write("""library(cricketdata)

ids  <- eval(parse(text = readLines("cricinfo_ids.txt")))
cat("fetching", length(ids), "players\\n")

meta <- fetch_player_meta(ids)
write.csv(meta, "bowler_meta_all.csv", row.names = FALSE)
cat(nrow(meta), "written to bowler_meta_all.csv\\n")
""")

print("fetch_styles.R written\n")
print("Now run in a terminal:   Rscript fetch_styles.R")
print("Then continue with the next cell.")

fetch_styles.R written

Now run in a terminal:   Rscript fetch_styles.R
Then continue with the next cell.


Group the styles into six categories. **Two decisions:** a googly is a variation within leg
spin, and "fast-medium" and "medium-fast" are the same thing written inconsistently.

The `assert` halts the notebook if a style appears that isn't in the map, so nothing slips
through unclassified.

In [60]:
if not os.path.exists("bowler_meta_all.csv"):
    raise FileNotFoundError(
        "bowler_meta_all.csv not found. Run 'Rscript fetch_styles.R' first.\n"
        "For a different competition, regenerate cricinfo_ids.txt and re-run it."
    )

meta = pd.read_csv("bowler_meta_all.csv")

STYLE_MAP = {
    "Left-arm fast":          ("LF",  "Left-arm pace"),
    "Left-arm fast-medium":   ("LFM", "Left-arm pace"),
    "Left-arm medium":        ("LM",  "Left-arm pace"),
    "Left-arm medium-fast":   ("LFM", "Left-arm pace"),
    "Left-arm wrist-spin":    ("SLC", "Left-arm wrist spin"),
    "Legbreak":               ("LB",  "Leg spin"),
    "Legbreak googly":        ("LB",  "Leg spin"),     # googly is a variation  §6
    "Right-arm fast":         ("RF",  "Right-arm pace"),
    "Right-arm fast-medium":  ("RFM", "Right-arm pace"),
    "Right-arm medium":       ("RM",  "Right-arm pace"),
    "Right-arm medium-fast":  ("RFM", "Right-arm pace"),
    "Right-arm offbreak":     ("OB",  "Off spin"),
    "Slow left-arm orthodox": ("SLA", "Left-arm orthodox"),
}

unmapped = set(meta['bowling_style'].dropna()) - set(STYLE_MAP)
assert not unmapped, f"a style appeared that is not in STYLE_MAP: {unmapped}"

meta['style_code']  = meta['bowling_style'].map(lambda s: STYLE_MAP[s][0])
meta['style_group'] = meta['bowling_style'].map(lambda s: STYLE_MAP[s][1])
print(f"{len(meta)} bowlers classified into {meta.style_group.nunique()} groups")

372 bowlers classified into 6 groups


Join back to our player names and load. The view is dropped first, because it depends on this table.

In [61]:
link  = people[['key_cricinfo','identifier']].dropna()
names = query("SELECT registry_id, player_name FROM dim_player")

out = (meta.merge(link,  left_on='cricinfo_id', right_on='key_cricinfo')
           .merge(names, left_on='identifier',  right_on='registry_id')
           [['player_name','style_code','style_group']]
           .drop_duplicates())

assert len(out) > 0, "join produced no rows - regenerate the metadata file"

run_sql("DROP VIEW IF EXISTS v_bowler_type;")   # the view depends on this table
out.to_sql("bowler_style", engine, if_exists="replace", index=False)

check("bowlers in the database", int(scalar("SELECT COUNT(*) FROM bowler_style")),
      expected=len(all_bowlers))

query("SELECT * FROM bowler_style ORDER BY style_group, player_name LIMIT 8")

PASS  bowlers in the database: 372 (expected 372)


,player_name,style_code,style_group
0,AC Agar,SLA,Left-arm orthodox
1,AC Voges,SLA,Left-arm orthodox
2,AJ Finch,SLA,Left-arm orthodox
3,AJ Hosein,SLA,Left-arm orthodox
4,AK Heal,SLA,Left-arm orthodox
5,AP Devcich,SLA,Left-arm orthodox
6,AR Beadle,SLA,Left-arm orthodox
7,AW O'Brien,SLA,Left-arm orthodox


**Reality check on the mapping.** BBL should come back pace-dominated, with spin a minority
and left-arm wrist spin rare. More leg-spinners than pace bowlers would mean the mapping is
wrong, whatever the technical checks said.

In [62]:
query("""
SELECT s.style_group,
       COUNT(*) AS bowlers,
       SUM(x.balls) AS balls_bowled,
       ROUND(100.0*SUM(x.balls)/SUM(SUM(x.balls)) OVER (), 1) AS pct_of_deliveries
FROM bowler_style s
JOIN (SELECT p.player_name, COUNT(*) balls
      FROM fact_delivery f JOIN dim_player p ON f.bowler_id=p.registry_id
      GROUP BY p.player_name) x ON s.player_name = x.player_name
GROUP BY s.style_group
ORDER BY balls_bowled DESC
""")

,style_group,bowlers,balls_bowled,pct_of_deliveries
0,Right-arm pace,182,80033.0,52.2
1,Leg spin,47,20608.0,13.4
2,Left-arm pace,42,20567.0,13.4
3,Off spin,51,14680.0,9.6
4,Left-arm orthodox,37,12837.0,8.4
5,Left-arm wrist spin,13,4525.0,3.0


## 7 · Serving views

Four views, one per dashboard visual. **All aggregation happens here in SQL** — Power BI
imports these and only displays them, so the logic stays readable and testable outside a
`.pbix` file, and Power BI receives a few hundred rows instead of 153,250.

| View | Dashboard page | Answers |
|---|---|---|
| `v_scouting` | 1 — Batter plan | Which bowlers should we use against this batter? |
| `v_dismissals` | 1 — Batter plan | How does he get out? (drives field settings) |
| `v_bowler_type` | 2 — Bowling type | Is he weak against a *kind* of bowling? |
| `v_death_bowling` | 3 — Death overs | Who bowls the 19th? |

### The metric everything rests on

**`sr_ratio` = his strike rate against this bowler ÷ his own career strike rate.**

A raw strike rate of 104 tells you nothing on its own — is that good bowling, or a slow
batter? Aaron Finch's normal is 126, so 104 against one bowler means that bowler slows him
by a fifth. A batter whose normal *is* 104 shows no effect at all. The ratio separates the
bowler's effect from the batter's style. *(§7a)*

### 7a · `v_scouting` → dashboard page 1

This view is built in three stages, and it is worth seeing each one separately.

**Stage 1 — the batter's own normal.** Group every ball a batter has faced, across all
bowlers, and work out how he usually performs.

In [63]:
query(f"""
SELECT batter_id,
       COUNT(*) AS career_balls,
       ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr,
       ROUND(COUNT(*)::numeric /
             NULLIF(SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END),0),0) AS balls_per_dismissal,
       ROUND(100.0*SUM(CASE WHEN runs_off_bat IN (4,6) THEN 1 ELSE 0 END)/COUNT(*),1) AS boundary_pct,
       ROUND(100.0*SUM(CASE WHEN runs_off_bat=0 AND extras=0 THEN 1 ELSE 0 END)/COUNT(*),1) AS dot_pct
FROM fact_delivery
GROUP BY batter_id
HAVING COUNT(*) >= {BASELINE_MIN_BALLS}
ORDER BY career_sr DESC
LIMIT 5
""")

,batter_id,career_balls,career_sr,balls_per_dismissal,boundary_pct,dot_pct
0,1c513d54,563,151.0,22.0,19.2,28.4
1,f1f99156,957,148.8,22.0,18.7,29.2
2,b681e71e,2154,148.4,23.0,19.1,26.8
3,30a45b23,1016,146.2,34.0,17.3,25.8
4,45eda7c8,2812,145.4,25.0,20.2,34.6


**One row per batter, no bowler column.** This is the yardstick everything else gets
measured against.

Only batters with 500+ career balls qualify. A normal calculated from a handful of innings
is unreliable, and since every ratio below is divided by it, an unreliable yardstick makes
every ratio unreliable.

**Stage 2 — the same measures, but against one bowler at a time.**

In [64]:
query(f"""
SELECT batter_id, bowler_id,
       COUNT(*) AS balls,
       SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
       ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS matchup_sr,
       ROUND(100.0*SUM(CASE WHEN runs_off_bat IN (4,6) THEN 1 ELSE 0 END)/COUNT(*),1) AS boundary_pct,
       ROUND(100.0*SUM(CASE WHEN runs_off_bat=0 AND extras=0 THEN 1 ELSE 0 END)/COUNT(*),1) AS dot_pct
FROM fact_delivery
GROUP BY batter_id, bowler_id
HAVING COUNT(*) >= {MATCHUP_MIN_BALLS}
ORDER BY balls DESC
LIMIT 5
""")

,batter_id,bowler_id,balls,dismissals,matchup_sr,boundary_pct,dot_pct
0,45eda7c8,14f96089,98,5,91.8,11.2,52.0
1,1a156c88,bd93fea4,89,4,105.6,10.1,32.6
2,8bdac857,14f96089,87,2,109.2,8.0,27.6
3,b681e71e,e1b9f3a9,85,4,190.6,29.4,24.7
4,32198ae0,14f96089,81,2,129.6,11.1,24.7


**Now one row per pairing.** The only change is `GROUP BY batter_id, bowler_id` instead of
just `batter_id`.

**But these numbers still mean nothing on their own.** A strike rate of 106 could be
containment or could be that batter's ordinary rate — stage 2 has no access to the
baseline, so it cannot tell.

**Stage 3 — join the two and divide.** This is where a number becomes a finding.

In [65]:
run_sql(f"""
CREATE OR REPLACE VIEW v_scouting AS
WITH baseline AS (
    -- stage 1: the batter's own normal
    SELECT batter_id,
           COUNT(*) AS career_balls,
           ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr,
           ROUND(COUNT(*)::numeric /
                 NULLIF(SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END),0),0)
             AS balls_per_dismissal,
           ROUND(100.0*SUM(CASE WHEN runs_off_bat IN (4,6) THEN 1 ELSE 0 END)
                 /COUNT(*),1) AS career_boundary_pct,
           ROUND(100.0*SUM(CASE WHEN runs_off_bat=0 AND extras=0 THEN 1 ELSE 0 END)
                 /COUNT(*),1) AS career_dot_pct
    FROM fact_delivery GROUP BY batter_id
    HAVING COUNT(*) >= {BASELINE_MIN_BALLS}
),
matchup AS (
    -- stage 2: the same measures, per bowler
    SELECT batter_id, bowler_id,
           COUNT(*) AS balls,
           SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS matchup_sr,
           ROUND(100.0*SUM(CASE WHEN runs_off_bat IN (4,6) THEN 1 ELSE 0 END)
                 /COUNT(*),1) AS matchup_boundary_pct,
           ROUND(100.0*SUM(CASE WHEN runs_off_bat=0 AND extras=0 THEN 1 ELSE 0 END)
                 /COUNT(*),1) AS matchup_dot_pct
    FROM fact_delivery GROUP BY batter_id, bowler_id
    HAVING COUNT(*) >= {MATCHUP_MIN_BALLS}
)
-- stage 3: the join brings both numbers onto one row so they can be divided
SELECT bat.player_name  AS batter,
       b.career_sr, b.career_balls, b.balls_per_dismissal,
       b.career_boundary_pct, b.career_dot_pct,
       bowl.player_name AS bowler,
       m.balls, m.matchup_sr, m.dismissals,
       m.matchup_boundary_pct, m.matchup_dot_pct,
       ROUND(m.matchup_sr / b.career_sr, 2) AS sr_ratio,
       ROUND(m.matchup_dot_pct / NULLIF(b.career_dot_pct,0), 2) AS dot_ratio,
       ROUND(m.matchup_boundary_pct / NULLIF(b.career_boundary_pct,0), 2) AS boundary_ratio
FROM matchup m
JOIN baseline b      ON m.batter_id = b.batter_id
JOIN dim_player bat  ON m.batter_id = bat.registry_id
JOIN dim_player bowl ON m.bowler_id = bowl.registry_id;
""")
print("v_scouting created")

v_scouting created


**Validate.** A batter-bowler pairing needs at least 30 balls before it means anything.
Over 10 balls a batter might score 4 runs or 20 purely by chance, and either would look
like a pattern. The view enforces the 30-ball minimum and the dashboard footer states it,
so this check confirms the thinnest pairing really does clear 30.

In [66]:
s = query("SELECT COUNT(*) n, COUNT(DISTINCT batter) b, MIN(balls) mb FROM v_scouting").iloc[0]

check(f"thinnest pairing has {MATCHUP_MIN_BALLS}+ balls", int(s["mb"]),
      lo=MATCHUP_MIN_BALLS, hi=10000)
check("pairings available", int(s["n"]), lo=400, hi=2000)
check("batters covered",    int(s["b"]), lo=50,  hi=200)

PASS  thinnest pairing has 30+ balls: 30 (expected 30-10,000)
PASS  pairings available: 620 (expected 400-2,000)
PASS  batters covered: 73 (expected 50-200)


The finished view. Three ratios, because containment happens in different ways:

- **`sr_ratio`** — is he slowed down?
- **`dot_ratio`** — is he being starved of singles? *(above 1.00 = more dot balls than usual)*
- **`boundary_ratio`** — are his big shots being cut off? *(below 1.00 = fewer fours and sixes)*

A coach sets a different field for a bowler who denies singles than for one who denies
boundaries, and `sr_ratio` alone cannot tell them apart.

In [67]:
print("CA Lynn - the bowlers who suppress him most\n")
query("""SELECT bowler, balls, career_sr, matchup_sr,
                sr_ratio, dot_ratio, boundary_ratio, dismissals
         FROM v_scouting WHERE batter='CA Lynn' ORDER BY sr_ratio LIMIT 6""")

CA Lynn - the bowlers who suppress him most



,bowler,balls,career_sr,matchup_sr,sr_ratio,dot_ratio,boundary_ratio,dismissals
0,P Hatzoglou,33,145.4,57.6,0.40,1.05,0.00,0
1,SNJ O'Keefe,49,145.4,69.4,0.48,1.06,0.00,1
2,MG Neser,42,145.4,90.5,0.62,1.31,0.71,2
3,A Zampa,98,145.4,91.8,0.63,1.50,0.55,5
4,GB Hogg,33,145.4,93.9,0.65,0.88,0.30,2
5,MA Beer,53,145.4,96.2,0.66,1.20,0.56,1


### 7b · `v_dismissals` → dashboard page 1

**How a batter gets out drives the field setting.** Caught-heavy means catchers; bowled or
lbw means attack the stumps.

Run-outs are excluded, because no bowling plan can influence them.

In [68]:
run_sql("""
CREATE OR REPLACE VIEW v_dismissals AS
SELECT bat.player_name AS batter, f.wicket_type, COUNT(*) AS dismissals
FROM fact_delivery f
JOIN dim_player bat ON f.batter_id = bat.registry_id
WHERE f.bowler_wicket = TRUE
GROUP BY bat.player_name, f.wicket_type;
""")

# the totals here must equal the bowler-credited wickets in the fact table
check("reconciles to the fact table",
      int(scalar("SELECT SUM(dismissals) FROM v_dismissals")),
      expected=int(scalar("SELECT COUNT(*) FROM fact_delivery WHERE bowler_wicket")))

query("""SELECT wicket_type, dismissals,
                ROUND(100.0*dismissals/SUM(dismissals) OVER (),0) AS pct
         FROM v_dismissals WHERE batter='CA Lynn' ORDER BY dismissals DESC""")

PASS  reconciles to the fact table: 7,637 (expected 7,637)


,wicket_type,dismissals,pct
0,caught,87,77.0
1,bowled,17,15.0
2,lbw,5,4.0
3,caught and bowled,3,3.0
4,stumped,1,1.0


### 7c · `v_bowler_type` → dashboard page 2

Page 1 tells a coach to bowl one particular bowler at one particular batter. That only
helps if you have that bowler. This view groups bowlers by **type** instead — leg spin, off
spin, right-arm pace — so the answer works against any opponent.

It also multiplies the evidence: one bowler against one batter is 30 to 90 balls; every
leg-spinner together is 400 or more.

Some rows rest on 400 balls, others on 80. Both show a ratio, and on screen they look
equally trustworthy. `sample_confidence` marks each row *solid* (200+ balls), *moderate*
(100–199) or *thin* (under 100), and the dashboard fades the thin ones so the eye goes to
the reliable numbers.

In [69]:
run_sql(f"""
CREATE OR REPLACE VIEW v_bowler_type AS
WITH baseline AS (
    SELECT batter_id, ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr
    FROM fact_delivery GROUP BY batter_id HAVING COUNT(*) >= {BASELINE_MIN_BALLS}
),
by_type AS (
    SELECT f.batter_id, s.style_group, COUNT(*) AS balls,
           SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) AS type_sr
    FROM fact_delivery f
    JOIN dim_player p   ON f.bowler_id = p.registry_id
    JOIN bowler_style s ON p.player_name = s.player_name
    GROUP BY f.batter_id, s.style_group
    HAVING COUNT(*) >= {TYPE_MIN_BALLS}
)
SELECT bat.player_name AS batter, t.style_group AS bowling_type,
       t.balls, t.dismissals, t.type_sr, b.career_sr,
       ROUND(t.type_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN t.balls >= 200 THEN 'solid'
            WHEN t.balls >= 100 THEN 'moderate'
            ELSE 'thin' END AS sample_confidence
FROM by_type t
JOIN baseline b     ON t.batter_id = b.batter_id
JOIN dim_player bat ON t.batter_id = bat.registry_id;
""")

check(f"all rows meet the {TYPE_MIN_BALLS}-ball minimum",
      int(scalar("SELECT MIN(balls) FROM v_bowler_type")), lo=TYPE_MIN_BALLS, hi=100000)

query("SELECT * FROM v_bowler_type WHERE batter='CA Lynn' ORDER BY sr_ratio")

PASS  all rows meet the 60-ball minimum: 60 (expected 60-100,000)


,batter,bowling_type,balls,dismissals,type_sr,career_sr,sr_ratio,sample_confidence
0,CA Lynn,Leg spin,434,21,106.0,145.4,0.73,solid
1,CA Lynn,Left-arm orthodox,328,8,106.4,145.4,0.73,solid
2,CA Lynn,Left-arm wrist spin,124,5,119.4,145.4,0.82,moderate
3,CA Lynn,Left-arm pace,340,12,141.5,145.4,0.97,solid
4,CA Lynn,Right-arm pace,1404,55,166.7,145.4,1.15,solid
5,CA Lynn,Off spin,182,12,169.8,145.4,1.17,moderate


**Read that result carefully.** The obvious conclusion is "he struggles against spin" — but
off spin is his *best* type while leg spin is his worst, and both are spin.

The pattern is **which way the ball turns**. Lynn is right-handed. Leg spin and left-arm
orthodox both turn away from him; off spin turns into him. So the finding is not spin
versus pace — it is the ball that leaves the bat.

That is a cricket insight rather than a statistical one. No model would have labelled it,
because it needs someone to notice that two of the six categories share a physical
property.

### 7d · `v_death_bowling` → dashboard page 3

Pages 1 and 2 ask who to bowl at a batter. This one asks something else: **who bowls the
19th over?** The last five overs decide T20 matches, and that call gets made in seconds.

**Economy** is runs conceded per over. Extras are included, because a wide costs the
bowling side a run whoever bowled it.

**Dot ball %** is the share of deliveries that concede nothing. It matters because a batter
who cannot score for three balls tends to take a risk on the fourth — dots create the
wicket, even when the dot-baller is not the one who takes it.

⚠️ **`team_ref` is a rough guess at each bowler's team.** The data never says which club a
player belongs to, so the team is taken from whichever side bowled first in his matches.
That works most of the time, but a player who moved clubs will appear twice, once under
each. Use it to narrow a long list down to roughly one squad — not to answer "who plays for
the Strikers".

In [70]:
run_sql("""
CREATE OR REPLACE VIEW v_death_bowling AS
SELECT p.player_name AS bowler,
       m.team_bowling_first AS team_ref,      -- rough, see the note above  §7d
       COUNT(*) AS balls,
       SUM(f.runs_off_bat + f.extras) AS runs_conceded,
       SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS wickets,
       ROUND(6.0*SUM(f.runs_off_bat + f.extras)/COUNT(*),2) AS economy,
       ROUND(100.0*SUM(CASE WHEN f.runs_off_bat=0 AND f.extras=0 THEN 1 ELSE 0 END)
             /COUNT(*),1) AS dot_ball_pct
FROM fact_delivery f
JOIN dim_player p ON f.bowler_id = p.registry_id
JOIN dim_match m  ON f.match_id  = m.match_id
WHERE f.phase = 'Death'
GROUP BY p.player_name, m.team_bowling_first
HAVING COUNT(*) >= 60;
""")

d = query("SELECT MIN(economy) lo, MAX(economy) hi FROM v_death_bowling").iloc[0]
check("best death economy plausible",  float(d["lo"]), lo=3.0, hi=12.0)
check("worst death economy plausible", float(d["hi"]), lo=8.0, hi=20.0)

query("SELECT * FROM v_death_bowling ORDER BY economy LIMIT 10")

PASS  best death economy plausible: 5.1 (expected 3.0-12.0)
PASS  worst death economy plausible: 12.19 (expected 8.0-20.0)


,bowler,team_ref,balls,runs_conceded,wickets,economy,dot_ball_pct
0,SL Malinga,Melbourne Stars,60,51,5,5.10,41.7
1,AJ Tye,Brisbane Heat,60,52,9,5.20,58.3
2,Mujeeb Ur Rahman,Brisbane Heat,73,71,10,5.84,45.2
3,AC Agar,Perth Scorchers,110,113,9,6.16,41.8
4,KW Richardson,Sydney Thunder,68,75,8,6.62,42.6
5,T Sangha,Sydney Thunder,66,75,8,6.82,36.4
6,BW Hilfenhaus,Melbourne Stars,83,96,10,6.94,31.3
7,B Laughlin,Melbourne Stars,66,78,5,7.09,27.3
8,Mohammad Nabi,Melbourne Renegades,60,71,5,7.10,28.3
9,GB Hogg,Perth Scorchers,80,95,7,7.13,30.0


---

## Pipeline complete

Four views ready for Power BI. Every stage validated against the stage before it, and
against cricket itself.

**Power BI connection, page layouts and the relationship trap** — `METHODOLOGY.md` §8.
**Limitations and scope** — `METHODOLOGY.md` §9.

In [71]:
query("SELECT ROUND(AVG(economy),2) AS avg_economy FROM v_death_bowling")

,avg_economy
0,9.05
